# Fine-tuning Dad's Model — QLoRA on Apple Silicon

This notebook fine-tunes a small language model on Dr. George Calhoun's Forbes corpus using **LoRA on MLX** — the Apple Silicon equivalent of QLoRA.

> **Why MLX instead of bitsandbytes QLoRA?**  
> `bitsandbytes` 4-bit quantization requires CUDA. On Apple Silicon (M4 Mac mini), we use Apple's `mlx-lm` library instead. It supports the same quantize-then-LoRA approach (quantize base model → train small LoRA adapter) which is functionally identical to QLoRA — just Metal instead of CUDA.

## Pipeline
```
train.jsonl / heldout.jsonl  →  mlx train.jsonl / valid.jsonl  →  LoRA fine-tuning (mlx-lm)
   (26a leakage-free split)                                              ↓
                                            eval: perplexity + sample generations
                                                          ↓
                                        conductor comparison (base vs fine-tuned)
```

**Config & data prep:** centralised in `training/finetune_config.py` (`QLoRAConfig` +
`prepare_mlx_data`), unit-tested in `tests/test_finetune_config.py`, so the run is
reproducible.  
**Model:** `microsoft/Phi-3-mini-4k-instruct` (3.8B — matches `phi3:mini` in your Ollama roster). Swap via `QLoRAConfig.for_base(...)`.  
**Data:** `data/training/{train,heldout}.jsonl` — the deterministic, **#25-eval-safe** split from `training/prepare.py` (plan 0008 step 26a). Training never sees an article a faithfulness-eval question is grounded in.  
**Est. training time:** ~10–30 min on M4 for 200 iters

## 0 — Environment Check

In [ ]:
import platform, subprocess, sys, os

print(f"Python:   {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Machine:  {platform.machine()}")

# Apple Silicon check
is_apple_silicon = platform.machine() == 'arm64'
print(f"\nApple Silicon: {is_apple_silicon}")
if not is_apple_silicon:
    print("⚠️  Not Apple Silicon — you could use bitsandbytes QLoRA instead (see comments in training cell)")

# Check MLX
try:
    import mlx.core as mx
    print(f"MLX:      {mx.__version__} ✓")
except ImportError:
    print("MLX:      not installed — run cell below to install")

# Check mlx-lm
try:
    import mlx_lm
    print(f"mlx-lm:   installed ✓")
except ImportError:
    print("mlx-lm:   not installed — run cell below to install")

In [ ]:
# Install dependencies (run once)
# mlx-lm handles everything: MLX, LoRA trainer, generation, quantization
!pip install mlx-lm matplotlib pandas --quiet

# For conductor comparison later
!pip install openai --quiet

print("✓ Dependencies installed")

## 1 — Config

In [ ]:
from pathlib import Path
import sys, json

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT = Path("../").resolve()          # adjust if running notebook from elsewhere
sys.path.insert(0, str(REPO_ROOT))         # make the `training` package importable

# Single source of truth for config + leakage-safe data prep (plan 0008 step 26c).
# Unit-tested in tests/test_finetune_config.py.
from training.finetune_config import QLoRAConfig, SMALL_BASES, TRAINING_DIR, FINETUNE_DIR

MANIFEST_PATH    = REPO_ROOT / "data" / "manifest.json"
LINGUISTICS_PATH = REPO_ROOT / "data" / "analysis" / "linguistics.json"
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)   # where adapter weights are saved

# ── Config ───────────────────────────────────────────────────────────────────
# Swap the base by alias, e.g. QLoRAConfig.for_base("qwen2.5-3b", train_iters=400).
# Available aliases: see SMALL_BASES.
cfg = QLoRAConfig()

BASE_MODEL    = cfg.base_model
LORA_LAYERS   = cfg.lora_layers
LORA_RANK     = cfg.lora_rank
TRAIN_ITERS   = cfg.train_iters
BATCH_SIZE    = cfg.batch_size
LEARNING_RATE = cfg.learning_rate
MAX_SEQ_LEN   = cfg.max_seq_len
CONDUCTOR_URL = cfg.conductor_url

print("Config (from training/finetune_config.QLoRAConfig):")
print(f"  Base model:    {BASE_MODEL}")
print(f"  Alternatives:  {', '.join(SMALL_BASES)}")
print(f"  LoRA rank:     {LORA_RANK}  layers: {LORA_LAYERS}")
print(f"  Train iters:   {TRAIN_ITERS}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Max seq len:   {MAX_SEQ_LEN}")
print(f"  Output dir:    {FINETUNE_DIR}")

## 2 — Prepare Training Data

In [ ]:
import subprocess
from training.finetune_config import load_jsonl

train_path   = TRAINING_DIR / "train.jsonl"
heldout_path = TRAINING_DIR / "heldout.jsonl"

# Generate 26a's leakage-free split if it isn't there yet (writes train/heldout.jsonl).
if not (train_path.exists() and heldout_path.exists()):
    print("26a split not found — generating via `python -m training`...")
    result = subprocess.run(
        ["python3", "-m", "training"],
        cwd=REPO_ROOT, capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)

# Load the deterministic, #25-eval-safe split produced by training/prepare.py (26a).
train_records   = load_jsonl(train_path)
heldout_records = load_jsonl(heldout_path)
examples = train_records + heldout_records   # combined, only for the stats/plots below

print(f"Loaded {len(train_records)} train + {len(heldout_records)} held-out examples (26a split)")

# Show one example
ex = examples[0]
print(f"\nExample structure: {len(ex['messages'])} messages")
for m in ex["messages"]:
    preview = m['content'][:120].replace('\n', ' ')
    print(f"  [{m['role']:9}] {preview}...")

In [ ]:
import pandas as pd

# Quick stats on the training data
word_counts = []
titles = []
for ex in examples:
    assistant_text = next((m['content'] for m in ex['messages'] if m['role'] == 'assistant'), '')
    user_text = next((m['content'] for m in ex['messages'] if m['role'] == 'user'), '')
    word_counts.append(len(assistant_text.split()))
    titles.append(user_text.replace('Write an analysis of ', '').rstrip('.'))

wc = pd.Series(word_counts)
print("Article word counts:")
print(wc.describe().round(0).astype(int).to_string())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(word_counts, bins=30, color='#c9a84c', edgecolor='#141414', alpha=0.85)
ax.set_xlabel('Words per article')
ax.set_ylabel('Count')
ax.set_title('Training Data: Article Length Distribution')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
print(f"\nTotal training tokens (rough): ~{sum(word_counts) * 1.3:.0f}")

In [ ]:
from training.finetune_config import prepare_mlx_data, eval_prompts

# Stage mlx-lm's expected train.jsonl / valid.jsonl from 26a's leakage-free split.
# No random re-shuffle: training uses exactly the deterministic, #25-eval-safe split,
# and validation is the held-out set (which excludes any article a #25 eval question
# is grounded in). This is the reproducibility fix at the heart of step 26c.
counts = prepare_mlx_data(TRAINING_DIR, FINETUNE_DIR)
n_train, n_valid = counts["n_train"], counts["n_valid"]

print(f"Train: {n_train} examples → {FINETUNE_DIR/'train.jsonl'}")
print(f"Valid (held-out): {n_valid} examples → {FINETUNE_DIR/'valid.jsonl'}")

# Deterministic held-out prompts for the generation comparison (§7).
EVAL_PROMPTS = eval_prompts(heldout_records, n=5)
print(f"\nEval prompts: {len(EVAL_PROMPTS)}")
for i, p in enumerate(EVAL_PROMPTS):
    print(f"  {i+1}. {p}")

## 3 — Baseline Generation (Before Fine-tuning)

Use the conductor to query `phi3:mini` before any fine-tuning. This gives us our comparison baseline.

In [ ]:
from openai import OpenAI
import textwrap

SYSTEM_PROMPT = (
    "You are Dr. George Calhoun, a Forbes technology analyst and academic. "
    "You write incisive, data-driven analysis on telecommunications, semiconductors, "
    "technology policy, ESG, nuclear energy, and the intersection of economics and innovation. "
    "Your style is rigorous but accessible, often contrarian, and grounded in evidence. "
    "Write in first person, as Dr. Calhoun would."
)

def query_conductor(prompt, model="phi3:mini", max_tokens=400, system=SYSTEM_PROMPT):
    """Query the conductor service and return the response text."""
    client = OpenAI(base_url=CONDUCTOR_URL, api_key="local")
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"[conductor error: {e}]"

print(f"Testing conductor at {CONDUCTOR_URL}...")
test = query_conductor(EVAL_PROMPTS[0], max_tokens=150)
if "conductor error" in test:
    print(f"⚠️  Conductor not reachable: {test}")
    print("   Start it with: cd /Volumes/FamilyWorkDrive/development/local-llm-conductor && ./start.sh")
    CONDUCTOR_AVAILABLE = False
else:
    print("✓ Conductor reachable")
    CONDUCTOR_AVAILABLE = True

In [ ]:
baseline_outputs = {}

if CONDUCTOR_AVAILABLE:
    print("Generating baseline outputs from phi3:mini (pre-fine-tune)...\n")
    print("=" * 70)
    for prompt in EVAL_PROMPTS[:3]:  # first 3 prompts to keep it quick
        output = query_conductor(prompt, max_tokens=300)
        baseline_outputs[prompt] = output
        print(f"PROMPT: {prompt}")
        print()
        for line in textwrap.wrap(output, width=70):
            print(f"  {line}")
        print("=" * 70)
        print()
else:
    print("Skipping baseline — conductor not available. Fine-tuning will still run.")

## 4 — LoRA Fine-tuning

Uses `mlx_lm.lora` — Apple Silicon native training. The `--quantize` flag replicates QLoRA: base model is 4-bit quantized in-memory, LoRA adapters trained in fp16.

**Adapter output:** `data/finetune_run/adapters/`  
**Training log:** streamed to stdout (watch for `val loss` dropping)

In [ ]:
import subprocess, sys, time

# Build the mlx_lm.lora command
# Docs: https://github.com/ml-explore/mlx-lm
cmd = [
    sys.executable, "-m", "mlx_lm.lora",
    "--model",        BASE_MODEL,
    "--train",
    "--data",         str(FINETUNE_DIR),
    "--adapter-path", str(FINETUNE_DIR / "adapters"),
    "--iters",        str(TRAIN_ITERS),
    "--batch-size",   str(BATCH_SIZE),
    "--lora-layers",  str(LORA_LAYERS),
    "--lora-rank",    str(LORA_RANK),
    "--learning-rate", str(LEARNING_RATE),
    "--max-seq-length", str(MAX_SEQ_LEN),
    "--steps-per-eval", "25",    # validate every 25 steps
    "--save-every",   "100",     # save checkpoint every 100 steps
    "--quantize",                # 4-bit quantize base model (= QLoRA on Apple Silicon)
    "--val-batches",  "-1",      # use all validation data
]

print("Starting LoRA fine-tuning...")
print("Command: " + " ".join(cmd))
print()
print("(This will take ~10–30 min depending on iters. Watch val loss ↓)")
print("=" * 70)

train_log = []
start_time = time.time()

# Stream output line by line so you see progress in the notebook
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    line = line.rstrip()
    print(line)
    train_log.append(line)

process.wait()
elapsed = time.time() - start_time

print("=" * 70)
if process.returncode == 0:
    print(f"✓ Training complete in {elapsed/60:.1f} min")
else:
    print(f"✗ Training failed (exit {process.returncode}) after {elapsed/60:.1f} min")

## 5 — Training Loss Curve

In [ ]:
import re

# Parse training log for loss values
# mlx-lm log format: "Iter N: Train loss X.XXX, Learning Rate Y.YYY, It/sec Z.ZZ"
# and               "Iter N: Val loss X.XXX, Val took Z.Zs"

train_steps, train_losses = [], []
val_steps, val_losses = [], []

for line in train_log:
    # Train loss
    m = re.search(r'Iter\s+(\d+):.*?[Tt]rain\s+loss\s+([\d.]+)', line)
    if m:
        train_steps.append(int(m.group(1)))
        train_losses.append(float(m.group(2)))
    # Val loss
    m = re.search(r'Iter\s+(\d+):.*?[Vv]al\s+loss\s+([\d.]+)', line)
    if m:
        val_steps.append(int(m.group(1)))
        val_losses.append(float(m.group(2)))

if not train_steps:
    print("No loss data parsed — check training output above.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Training loss
    axes[0].plot(train_steps, train_losses, color='#6ba3d6', linewidth=1.5, alpha=0.8, label='Train loss')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)

    # Validation loss
    if val_steps:
        axes[1].plot(val_steps, val_losses, color='#c9a84c', linewidth=2, marker='o', markersize=4, label='Val loss')
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Validation Loss')
        axes[1].spines['top'].set_visible(False)
        axes[1].spines['right'].set_visible(False)

        # Annotate improvement
        improvement = val_losses[0] - val_losses[-1]
        axes[1].annotate(
            f"Δ {improvement:.3f}",
            xy=(val_steps[-1], val_losses[-1]),
            xytext=(val_steps[-1] * 0.6, val_losses[-1] + improvement * 0.3),
            arrowprops=dict(arrowstyle='->', color='#c9a84c'),
            fontsize=10, color='#c9a84c'
        )
    else:
        axes[1].text(0.5, 0.5, 'No validation data', ha='center', va='center',
                     transform=axes[1].transAxes, color='gray')

    plt.suptitle('LoRA Fine-tuning — Loss Curves', fontsize=13)
    plt.tight_layout()
    plt.savefig(FINETUNE_DIR / 'loss_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nFinal train loss: {train_losses[-1]:.4f}")
    if val_losses:
        print(f"Final val loss:   {val_losses[-1]:.4f}")
        print(f"Val perplexity:   {2**val_losses[-1]:.2f}  (lower = better)")
        print(f"Improvement:      {val_losses[0] - val_losses[-1]:.4f} loss ({(val_losses[0]-val_losses[-1])/val_losses[0]*100:.1f}%)")

## 6 — Perplexity Evaluation

**Perplexity = exp(cross-entropy loss)**. Lower = the model is less surprised by the validation text = it's learned to write more like Dr. Calhoun.

We compute it on the held-out validation set for both base and fine-tuned model.

In [ ]:
import math

def run_mlx_eval(model_path, adapter_path=None, data_dir=None):
    """Run mlx_lm.lora --evaluate and parse val loss."""
    cmd = [
        sys.executable, "-m", "mlx_lm.lora",
        "--model",     model_path,
        "--evaluate",
        "--data",      str(data_dir or FINETUNE_DIR),
        "--batch-size", "1",
        "--max-seq-length", str(MAX_SEQ_LEN),
        "--val-batches", "-1",
    ]
    if adapter_path and Path(adapter_path).exists():
        cmd += ["--adapter-path", str(adapter_path)]
        if Path(str(adapter_path)).exists() and any(Path(adapter_path).glob("*.npz")):
            cmd += ["--quantize"]  # keep same quantize flag as training

    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stdout + result.stderr

    # Parse val loss
    m = re.search(r'[Vv]al\s+loss\s+([\d.]+)', output)
    if m:
        loss = float(m.group(1))
        return loss, math.exp(loss), output
    return None, None, output


adapter_dir = FINETUNE_DIR / "adapters"

print("Evaluating BASE model (no adapters)...")
base_loss, base_ppl, base_out = run_mlx_eval(BASE_MODEL)

print("Evaluating FINE-TUNED model (with adapters)...")
ft_loss, ft_ppl, ft_out = run_mlx_eval(BASE_MODEL, adapter_path=adapter_dir)

print()
print("┌─────────────────────────────────────────┐")
print("│           Perplexity Results            │")
print("├──────────────────┬────────────┬─────────┤")
print("│ Model            │ Val Loss   │ PPL     │")
print("├──────────────────┼────────────┼─────────┤")
if base_loss is not None:
    print(f"│ Base (phi3:mini) │ {base_loss:10.4f} │ {base_ppl:7.2f} │")
else:
    print(f"│ Base (phi3:mini) │ eval failed│    n/a  │")
if ft_loss is not None:
    print(f"│ Fine-tuned       │ {ft_loss:10.4f} │ {ft_ppl:7.2f} │")
else:
    print(f"│ Fine-tuned       │ eval failed│    n/a  │")
print("└──────────────────┴────────────┴─────────┘")

if base_ppl and ft_ppl:
    delta = base_ppl - ft_ppl
    pct = delta / base_ppl * 100
    print(f"\nPPL improvement: {delta:.2f} ({pct:.1f}% reduction)")
    if pct > 20:
        print("Strong fit — model has clearly adapted to Dr. Calhoun's style.")
    elif pct > 5:
        print("Moderate fit — more iters or data would help further.")
    else:
        print("Weak fit — try more iters (500+) or a higher LoRA rank.")

## 7 — Sample Generations: Before vs After

In [ ]:
from mlx_lm import load, generate

def load_model_for_generation(model_path, adapter_path=None):
    """Load model (and optionally adapters) for generation."""
    kwargs = {}
    if adapter_path and Path(adapter_path).exists():
        kwargs['adapter_path'] = str(adapter_path)
    model, tokenizer = load(model_path, **kwargs)
    return model, tokenizer


def generate_response(model, tokenizer, user_prompt, max_tokens=250):
    """Apply chat template and generate a response."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]
    # Apply the model's chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    return generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens, verbose=False)


# Load base model
print(f"Loading base model: {BASE_MODEL}")
print("(First load downloads weights ~2–4GB — subsequent loads are cached)")
base_model, base_tokenizer = load_model_for_generation(BASE_MODEL)
print("✓ Base model loaded")

In [ ]:
# Load fine-tuned model (base + adapters)
print(f"Loading fine-tuned model (with adapters from {adapter_dir})")
ft_model, ft_tokenizer = load_model_for_generation(BASE_MODEL, adapter_path=adapter_dir)
print("✓ Fine-tuned model loaded")

In [ ]:
# Generate and compare on eval prompts
comparisons = []

for prompt in EVAL_PROMPTS[:3]:
    print(f"\n{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")

    base_out = generate_response(base_model, base_tokenizer, prompt, max_tokens=250)
    ft_out   = generate_response(ft_model, ft_tokenizer, prompt, max_tokens=250)

    comparisons.append({"prompt": prompt, "base": base_out, "finetuned": ft_out})

    print("\n── BASE (phi3:mini, no fine-tuning) ──")
    for line in textwrap.wrap(base_out.strip(), 68):
        print(f"  {line}")

    print("\n── FINE-TUNED (dad-style LoRA) ──")
    for line in textwrap.wrap(ft_out.strip(), 68):
        print(f"  {line}")

print(f"\n{'='*70}")

## 8 — Style Similarity Heuristics

A few quick quantitative checks: does the fine-tuned model write more like Dr. Calhoun?

In [ ]:
import json
from training.finetune_config import style_metrics

# style_metrics(text, distinctive_words) lives in training/finetune_config.py — shared with
# the 26d voice eval and unit-tested in tests/test_finetune_config.py.

# Load distinctive words from linguistic analysis as the "Calhoun fingerprint"
calhoun_words = set()
if LINGUISTICS_PATH.exists():
    ling = json.loads(LINGUISTICS_PATH.read_text())
    calhoun_words = {w['word'] for w in ling.get('distinctive_words', [])[:30]}
    print(f"Loaded {len(calhoun_words)} distinctive Calhoun words from linguistics analysis")
    print(f"Sample: {sorted(calhoun_words)[:10]}")
else:
    print("Linguistics data not found — run `python -m analysis linguistic` first")


if calhoun_words and comparisons:
    print("\nStyle metric comparison (fine-tuned vs base):")
    print(f"{'Metric':<28} {'Base':>8}  {'Fine-tuned':>10}  {'Δ':>8}")
    print("-" * 60)

    # Aggregate across all comparison outputs
    all_base = " ".join(c['base'] for c in comparisons)
    all_ft   = " ".join(c['finetuned'] for c in comparisons)

    base_m = style_metrics(all_base, calhoun_words)
    ft_m   = style_metrics(all_ft, calhoun_words)

    for key in base_m:
        bv, fv = base_m[key], ft_m[key]
        delta = fv - bv
        print(f"  {key:<26} {bv:>8}  {fv:>10}  {delta:>+8.1f}")

    # Also show what the actual corpus looks like
    print()
    if LINGUISTICS_PATH.exists():
        agg = ling.get('aggregate', {})
        print(f"  Corpus avg sentence length: {agg.get('avg_sentence_length', 'n/a')}")
        print(f"  Corpus avg TTR:             {agg.get('avg_type_token_ratio', 'n/a')}")

## 9 — Conductor Comparison

Side-by-side: conductor's `phi3:mini` (stock, pulled from Ollama) vs our fine-tuned model running directly through MLX.

In [ ]:
# One final comparison on a fresh prompt not in training data
FRESH_PROMPTS = [
    "Write an analysis of the future of 5G spectrum policy in the United States.",
    "Write an analysis of whether nuclear power deserves a second look in the ESG era.",
]

for fp in FRESH_PROMPTS[:1]:   # just one to keep output manageable
    print(f"PROMPT: {fp}")
    print()

    # Via conductor (stock phi3:mini)
    if CONDUCTOR_AVAILABLE:
        conductor_out = query_conductor(fp, model="phi3:mini", max_tokens=300)
        print("── CONDUCTOR (stock phi3:mini) ──")
        for line in textwrap.wrap(conductor_out.strip(), 68):
            print(f"  {line}")
        print()

    # Via fine-tuned model (mlx)
    mlx_out = generate_response(ft_model, ft_tokenizer, fp, max_tokens=300)
    print("── FINE-TUNED (dad LoRA via MLX) ──")
    for line in textwrap.wrap(mlx_out.strip(), 68):
        print(f"  {line}")
    print()

## 10 — Save Results Summary

In [ ]:
from datetime import datetime

summary = {
    "timestamp": datetime.now().isoformat(),
    "base_model": BASE_MODEL,
    # Embed the exact config object that drove this run (single source of truth),
    # so the summary is provably reproducible.
    "config": cfg.to_dict(),
    "data": {
        "split": "26a leakage-free (train.jsonl / heldout.jsonl)",
        "n_train": n_train,
        "n_valid": n_valid,
    },
    "eval": {
        "base_val_loss": base_loss,
        "base_perplexity": round(base_ppl, 2) if base_ppl else None,
        "finetuned_val_loss": ft_loss,
        "finetuned_perplexity": round(ft_ppl, 2) if ft_ppl else None,
        "ppl_improvement_pct": round((base_ppl - ft_ppl) / base_ppl * 100, 1) if base_ppl and ft_ppl else None,
    },
    "final_train_loss": train_losses[-1] if train_losses else None,
    "comparisons": comparisons,
}

summary_path = FINETUNE_DIR / "run_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Summary saved to {summary_path}")
print()
print("=" * 50)
print("FINE-TUNING RUN COMPLETE")
print("=" * 50)
print(f"  Model:         {BASE_MODEL}")
print(f"  Adapters:      {adapter_dir}")
print(f"  Loss curves:   {FINETUNE_DIR / 'loss_curves.png'}")
print(f"  Summary:       {summary_path}")
if base_ppl and ft_ppl:
    print(f"  PPL:  {base_ppl:.1f} → {ft_ppl:.1f}  ({(base_ppl-ft_ppl)/base_ppl*100:.1f}% ↓)")

## 11 — (Optional) Fuse & Export to Ollama

Merge the LoRA adapters back into the base model weights and create a Modelfile for Ollama. After this, you can run your fine-tuned dad-model directly via `ollama run digital-dad`.

In [ ]:
# Fuse adapters into base model weights
FUSED_MODEL_DIR = FINETUNE_DIR / "fused_model"

print("Fusing LoRA adapters into base model...")
fuse_result = subprocess.run([
    sys.executable, "-m", "mlx_lm.fuse",
    "--model",        BASE_MODEL,
    "--adapter-path", str(adapter_dir),
    "--save-path",    str(FUSED_MODEL_DIR),
    "--quantize",     # keep 4-bit for smaller file size
    "--upload-repo",  "no",  # don't push to HuggingFace Hub
], capture_output=True, text=True)

print(fuse_result.stdout)
if fuse_result.returncode == 0:
    print(f"✓ Fused model saved to {FUSED_MODEL_DIR}")
else:
    print(f"✗ Fuse failed: {fuse_result.stderr}")

In [ ]:
# Convert fused MLX model → GGUF → create Ollama Modelfile
# Note: This requires llama.cpp's convert_hf_to_gguf.py
# For now, write the Modelfile template so you can adapt it manually.

OLLAMA_NAME = "digital-dad"

modelfile_content = f"""# Ollama Modelfile for digital-dad (Dr. George Calhoun LoRA)
# Generated by finetune_qlora.ipynb
#
# To build:
#   1. Convert fused MLX weights to GGUF:
#      python3 /path/to/llama.cpp/convert_hf_to_gguf.py {FUSED_MODEL_DIR} --outtype q4_0
#   2. Place the .gguf file next to this Modelfile
#   3. ollama create {OLLAMA_NAME} -f Modelfile
#   4. ollama run {OLLAMA_NAME}

FROM ./digital-dad.gguf

SYSTEM \""""{SYSTEM_PROMPT}\""""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_predict 1024
"""

modelfile_path = FINETUNE_DIR / "Modelfile"
modelfile_path.write_text(modelfile_content)
print(f"Modelfile written to {modelfile_path}")
print()
print(modelfile_content)